## Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)
**Document:** EU Artificial Intelligence Act (Regulation (EU) 2024/1689)

Two pipelines over the *same* chunked corpus:
1. **Baseline:** Bi-Encoder -> Top 10 (cosine) -> Top 3 straight to the LLM
2. **Re-ranking:** same Top 10 -> Cross-Encoder re-scores all 10 -> Top 3 re-ranked to the LLM



## 1. Setup

In [1]:
! pip install -q sentence-transformers faiss-cpu pypdf pandas google-genai openai anthropic

## 2. Config (change the LLM here and nowhere else)

| Provider | Example `LLM_MODEL` | Key env var |
|---|---|---|
| `gemini` | `gemini-2.5-flash` | `GEMINI_API_KEY` |
| `openai` | `gpt-4o-mini` | `OPENAI_API_KEY` |
| `anthropic` | any current Claude model id | `ANTHROPIC_API_KEY` |
| `ollama` | `llama3.1:8b` | none (local server) |

In Colab, add the key under the key icon (Secrets) with the same name as the env var.

In [2]:
# ---------------- LLM (swap freely) ----------------
LLM_PROVIDER = "gemini"            # "gemini" | "openai" | "anthropic" | "ollama"
LLM_MODEL    = "gemini-3.1-flash-lite-preview"  # model name for the chosen provider
LLM_TEMPERATURE = 0.0              # keep 0 so Answer 1 vs Answer 2 differ only because of the context

# ---------------- Retrieval models ----------------
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # Bi-Encoder
CROSS_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # Cross-Encoder

# ---------------- Document + chunking ----------------
PDF_PATH = "doc.pdf"   # upload the PDF next to the notebook
SKIP_RECITALS = True         # start from the operative text ("HAVE ADOPTED THIS REGULATION")
CHUNK_WORDS   = 140          # max words per chunk
CHUNK_OVERLAP = 30           # words of overlap between consecutive chunks of the same section

# ---------------- Retrieval ----------------
TOP_K_RETRIEVE = 10          # candidates from the Bi-Encoder
TOP_K_CONTEXT  = 3           # chunks given to the LLM

In [3]:
import os, re, time, textwrap
import numpy as np
import pandas as pd
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

# ---- API key handling (Colab secrets first, then environment) ----
KEY_ENV = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY",
           "anthropic": "ANTHROPIC_API_KEY", "ollama": None}

from dotenv import load_dotenv
import os

load_dotenv()

def get_api_key():
    name = KEY_ENV[LLM_PROVIDER]
    if name is None:
        return None
    return os.environ.get(name)



def call_llm(prompt: str) -> str:
    """Single entry point for the LLM. Both pipelines use this, so changing
    LLM_PROVIDER / LLM_MODEL in the config cell is all it takes to switch models."""
    key = get_api_key()
    if LLM_PROVIDER == "gemini":
        from google import genai
        client = genai.Client(api_key=key)
        r = client.models.generate_content(
            model=LLM_MODEL, contents=prompt,
            config={"temperature": LLM_TEMPERATURE})
        return r.text
    if LLM_PROVIDER == "openai":
        from openai import OpenAI
        r = OpenAI(api_key=key).chat.completions.create(
            model=LLM_MODEL, temperature=LLM_TEMPERATURE,
            messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if LLM_PROVIDER == "anthropic":
        import anthropic
        r = anthropic.Anthropic(api_key=key).messages.create(
            model=LLM_MODEL, max_tokens=800,
            messages=[{"role": "user", "content": prompt}])
        return r.content[0].text
    if LLM_PROVIDER == "ollama":
        import requests
        r = requests.post("http://localhost:11434/api/generate",
                          json={"model": LLM_MODEL, "prompt": prompt, "stream": False,
                                "options": {"temperature": LLM_TEMPERATURE}})
        return r.json()["response"]
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

c:\Users\DEDEEPYA\lma\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1019 13:48:16.105000 20464 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


## 3. Load the PDF and build chunks
Strategy: split on legal structure (`Article N` / `ANNEX X` headings) so a chunk never mixes two articles,
then cap long articles at `CHUNK_WORDS` words with a small overlap. Each chunk keeps its article label and page,
which makes failure analysis easy to read. Both pipelines use exactly this same corpus.

In [4]:
reader = PdfReader(PDF_PATH)
print("Pages:", len(reader.pages))

# Remove running headers/footers of the Official Journal layout
NOISE = re.compile(r"^(EN\s+)?OJ L, \d{1,2}\.\d{1,2}\.\d{4}(\s+EN)?\s*$|.*ELI: http.*")
pages = []
for i, p in enumerate(reader.pages, start=1):
    lines = [ln for ln in (p.extract_text() or "").splitlines() if not NOISE.match(ln.strip())]
    pages.append((i, "\n".join(lines)))

full = "\n".join(f"<<P{i}>>\n{t}" for i, t in pages)

if SKIP_RECITALS:
    m = re.search(r"HAVE ADOPTED THIS REGULATION", full)
    if m:
        full = full[m.end():]
    else:
        print("Marker not found; keeping the whole document.")

print("Characters:", len(full))

incorrect startxref pointer(1)
parsing for Object Streams


Pages: 144
Characters: 345330


In [5]:
PAGE_MARK = re.compile(r"<<P(\d+)>>")
HEADING = re.compile(r"(?m)^((?:Article \d+)|(?:ANNEX [IVXL]+))\s*$")

parts = HEADING.split(full)   # [pre, heading, body, heading, body, ...]
chunks = []                   # list of dicts: id, section, page, text
cur_page = 1

def add_section(label, body, start_page):
    words = PAGE_MARK.sub(" ", body).split()
    if not words:
        return
    # first line of the body is normally the article title
    title = body.strip().splitlines()[0].strip() if body.strip() else ""
    title = PAGE_MARK.sub("", title).strip()
    header = f"[{label} - {title}]" if title else f"[{label}]"
    step = CHUNK_WORDS - CHUNK_OVERLAP
    for s in range(0, len(words), step):
        piece = words[s:s + CHUNK_WORDS]
        if len(piece) < 30 and s > 0:   # drop tiny tail fragments
            break
        chunks.append({"id": len(chunks), "section": label, "page": start_page,
                       "text": header + " " + " ".join(piece)})
        if s + CHUNK_WORDS >= len(words):
            break

# text before the first heading (e.g. Chapter I intro) is skipped on purpose
for pm in PAGE_MARK.finditer(parts[0]):
    cur_page = int(pm.group(1))

for j in range(1, len(parts), 2):
    label, body = parts[j].strip(), parts[j + 1]
    add_section(label, body, cur_page)
    for pm in PAGE_MARK.finditer(body):
        cur_page = int(pm.group(1))

print("Total chunks:", len(chunks))
print("Avg words/chunk:", int(np.mean([len(c['text'].split()) for c in chunks])))
print("\nSample chunk:\n", textwrap.fill(chunks[5]["text"][:600], 110))

Total chunks: 509
Avg words/chunk: 136

Sample chunk:
 [Article 2 - Scope] in the Union, where the output is used in the Union exclusively for military, defence or
national security purposes, regardless of the type of entity carrying out those activities. 4. This Regulation
applies neither to public authorities in a third country nor to international organisations falling within the
scope of this Regulation pursuant to paragraph 1, where those authorities or organisations use AI systems in
the framework of international cooperation or agreements for law enforcement and judicial cooperation with the
Union or with one or more Member States, provided


## 4. Index with the Bi-Encoder

In [7]:
bi_encoder = SentenceTransformer(EMBED_MODEL)
t0 = time.perf_counter()
emb = bi_encoder.encode([c["text"] for c in chunks], batch_size=64,
                        normalize_embeddings=True, show_progress_bar=True).astype("float32")
print(f"Indexed {len(chunks)} chunks in {time.perf_counter()-t0:.1f}s, dim={emb.shape[1]}")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

cross_encoder = CrossEncoder(CROSS_MODEL)

Batches: 100%|██████████| 8/8 [00:01<00:00,  4.40it/s]


Indexed 509 chunks in 1.8s, dim=384


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2636.74it/s]


## 5. The two pipelines
Retrieval and re-ranking are timed separately from the LLM call so the latency table is clean.

In [8]:
def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Stage 1: Bi-Encoder -> top-k candidates (id, cosine score), best first."""
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

def rerank(query, candidates):
    """Stage 2: Cross-Encoder scores every (query, chunk) pair jointly, then re-sorts."""
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    out = [(cid, float(s)) for (cid, _), s in zip(candidates, ce_scores)]
    return sorted(out, key=lambda x: x[1], reverse=True)

PROMPT_TEMPLATE = """You are answering questions about the EU Artificial Intelligence Act.
Use ONLY the context below. If the context does not contain the answer, say so.
Mention any exceptions or conditions that appear in the context.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(query, chunk_ids):
    ctx = "\n\n".join(f"[Chunk {n+1}] {chunks[cid]['text']}" for n, cid in enumerate(chunk_ids))
    return PROMPT_TEMPLATE.format(context=ctx, question=query)   # identical for both pipelines

def run_pipelines(query, with_llm=True):
    res = {"query": query}
    t = time.perf_counter()
    cands = baseline_retrieve(query)
    res["t_retrieve"] = time.perf_counter() - t

    t = time.perf_counter()
    reranked = rerank(query, cands)
    res["t_rerank"] = time.perf_counter() - t

    res["baseline_rank"] = [cid for cid, _ in cands]           # ranks 1..10
    res["rerank_rank"]   = [cid for cid, _ in reranked]
    res["baseline_top3"] = res["baseline_rank"][:TOP_K_CONTEXT]
    res["rerank_top3"]   = res["rerank_rank"][:TOP_K_CONTEXT]
    res["bi_scores"] = dict(cands)
    res["ce_scores"] = dict(reranked)

    if with_llm:
        t = time.perf_counter(); res["answer1"] = call_llm(build_prompt(query, res["baseline_top3"])); res["t_llm1"] = time.perf_counter() - t
        t = time.perf_counter(); res["answer2"] = call_llm(build_prompt(query, res["rerank_top3"]));   res["t_llm2"] = time.perf_counter() - t
    return res

## Few test queries

In [9]:
def gold_rank(rank_list, gold_pattern):
    """1-based rank of the best-placed chunk matching gold_pattern, else None."""
    rx = re.compile(gold_pattern, re.I | re.S)
    for pos, cid in enumerate(rank_list, start=1):
        if rx.search(chunks[cid]["text"]):
            return pos
    return None

def analyse(category, query, gold_pattern, with_llm=False):
    r = run_pipelines(query, with_llm=with_llm)
    b = gold_rank(r["baseline_rank"], gold_pattern)
    c = gold_rank(r["rerank_rank"],  gold_pattern)
    r.update(category=category, gold_pattern=gold_pattern, gold_baseline_rank=b, gold_rerank_rank=c)
    return r    

## Failure cases:

In [10]:
FAILURE_CASES = [
        ("Exceptions", "When does the original provider not have to hand over documentation after a system becomes high-risk?",
        r"clearly specified that its AI system is not to be changed into a high-risk AI system"),
        ("Exceptions", "Can police filter a lawfully acquired image dataset by biometric data?",
        r"does not cover any labelling or filtering of lawfully acquired biometric datasets"),
        ## Article 5(1)(g) 
]


cand_runs, rows = [], []
for cat, q, g in FAILURE_CASES:
    r = analyse(cat, q, g, with_llm=False)
    cand_runs.append(r)
    b, c = r["gold_baseline_rank"], r["gold_rerank_rank"]
    rows.append({"category": cat, "query": q[:55],
                 "gold (baseline)": b or "not in top10",
                 "gold (re-rank)": c or "not in top10",
                 "PROMOTED into top3": bool(b and c and b > TOP_K_CONTEXT and c <= TOP_K_CONTEXT)})
cand_df = pd.DataFrame(rows)
cand_df

,category,query,gold (baseline),gold (re-rank),PROMOTED into top3
0,Exceptions,When does the original provider not have to ha...,8,1,True
1,Exceptions,Can police filter a lawfully acquired image da...,4,1,True


## Failure cases in multiple categories:

In [11]:
CANDIDATES = [
    ("Exceptions", "When is a system on the high-risk list not treated as high-risk?",
        r"By derogation from paragraph 2, an AI system referred to in Annex III shall not be considered"),
    ## Article 6 (3)
    ("Exceptions", "Do I have to label AI-generated videos of real people?",
        r"shall disclose that the content has been artificially generated or manipulated"),
    ## Article 50 (4)
    ("Document-specific", "What fines apply for violating the prohibited practices rules?",
                     r"(35 000 000|7 ?%)"),
    ## Article 99 (3)
    ("Keyword trap", "Who takes on the provider obligations if a deployer rebrands a high-risk system?",
        r"shall be considered to be a provider of a high-risk AI system"),
    ## Article 25 (1)(a)
    ("Negation", "Can a deployer act on a biometric identification without separate verification by two people?",
        r"separately verified and confirmed by at least two natural persons"),
    ## Article 14 (5)
]

cand_runs, rows = [], []
for cat, q, g in CANDIDATES:
    r = analyse(cat, q, g, with_llm=False)
    cand_runs.append(r)
    b, c = r["gold_baseline_rank"], r["gold_rerank_rank"]
    rows.append({"category": cat, "query": q[:55],
                 "gold (baseline)": b or "not in top10",
                 "gold (re-rank)": c or "not in top10",
                 "PROMOTED into top3": bool(b and c and b > TOP_K_CONTEXT and c <= TOP_K_CONTEXT)})
cand_df = pd.DataFrame(rows)
cand_df

,category,query,gold (baseline),gold (re-rank),PROMOTED into top3
0,Exceptions,When is a system on the high-risk list not tre...,5,1,True
1,Exceptions,Do I have to label AI-generated videos of real...,6,1,True
2,Document-specific,What fines apply for violating the prohibited ...,6,1,True
3,Keyword trap,Who takes on the provider obligations if a dep...,4,2,True
4,Negation,Can a deployer act on a biometric identificati...,6,1,True


### Inspect one query in detail

In [12]:
def show(r, n=10):
    print("="*100)
    print("QUERY:", r["query"], f"[{r.get('category','')}]\n")
    print("="*100)
    print(f"{'BaseRank':<9}{'ChunkID':<9}{'Section':<12}{'BiScore':<9}{'CE Score':<10}{'ReRank':<7}")
    rr = {cid: pos for pos, cid in enumerate(r["rerank_rank"], start=1)}
    for pos, cid in enumerate(r["baseline_rank"][:n], start=1):
        print(f"{pos:<9}{cid:<9}{chunks[cid]['section']:<12}{r['bi_scores'][cid]:<9.3f}{r['ce_scores'][cid]:<10.3f}{rr[cid]:<7}")
    if "answer1" in r:
        print("\n--- Answer 1 (baseline top 3) ---\n", r["answer1"])
        print("\n--- Answer 2 (re-ranked top 3) ---\n", r["answer2"])

# pick a row index from the summary table above
i = 0
full_run = analyse(*CANDIDATES[i], with_llm=True)
print("Gold chunk rank:", full_run["gold_baseline_rank"], "->", full_run["gold_rerank_rank"], "\n")
show(full_run)

Gold chunk rank: 5 -> 1 

QUERY: When is a system on the high-risk list not treated as high-risk? [Exceptions]

BaseRank ChunkID  Section     BiScore  CE Score  ReRank 
1        64       Article 9   0.659    1.630     5      
2        66       Article 9   0.637    -0.510    10     
3        65       Article 9   0.627    1.464     7      
4        69       Article 9   0.611    1.567     6      
5        51       Article 6   0.591    4.731     1      
6        68       Article 9   0.590    0.459     8      
7        67       Article 9   0.567    0.186     9      
8        89       Article 14  0.544    2.196     3      
9        50       Article 6   0.535    2.108     4      
10       106      Article 20  0.524    2.725     2      

--- Answer 1 (baseline top 3) ---
 The provided context does not contain information regarding when a system on the high-risk list is not treated as high-risk.

--- Answer 2 (re-ranked top 3) ---
 An AI system referred to in Annex III is not considered high-ri

### Every test query end-to-end

In [13]:
ALL_QUERIES = FAILURE_CASES + CANDIDATES

all_runs = [analyse(cat, q, g, with_llm=True) for cat, q, g in ALL_QUERIES]
for r in all_runs:
    print("=" * 100)
    show(r)

QUERY: When does the original provider not have to hand over documentation after a system becomes high-risk? [Exceptions]

BaseRank ChunkID  Section     BiScore  CE Score  ReRank 
1        103      Article 18  0.623    3.538     2      
2        104      Article 18  0.608    -2.323    9      
3        477      ANNEX VII   0.540    -7.973    10     
4        76       Article 11  0.509    -0.181    7      
5        95       Article 16  0.500    0.719     4      
6        121      Article 24  0.485    -0.033    6      
7        111      Article 22  0.473    1.148     3      
8        125      Article 25  0.468    4.562     1      
9        124      Article 25  0.466    0.502     5      
10       101      Article 17  0.464    -1.923    8      

--- Answer 1 (baseline top 3) ---
 Based on the provided context, the original provider is not explicitly excused from handing over documentation. However, the context specifies the following conditions regarding the availability of documentation:



## Latency profiling
Each stage is repeated `N_RUNS` times (after one warm-up) and averaged. LLM time is reported separately because it
is the same size for both pipelines and dominates end-to-end time.

In [14]:
N_RUNS = 10
bench_queries = [q for _, q, _ in ALL_QUERIES]
_ = run_pipelines(bench_queries[0], with_llm=False)   # warm-up

t_ret, t_rer = [], []
for _ in range(N_RUNS):
    for q in bench_queries:
        r = run_pipelines(q, with_llm=False)
        t_ret.append(r["t_retrieve"]); t_rer.append(r["t_rerank"])

llm_times = [(r["t_llm1"], r["t_llm2"]) for r in all_runs]

ret_ms = np.mean(t_ret) * 1000
rer_ms = np.mean(t_rer) * 1000

lat = pd.DataFrame({
    "Stage": ["Bi-encoder retrieval (embed query + FAISS top-10), used by both",
              "Cross-encoder re-rank of 10 pairs (Pipeline 2 only)",
              "Pipeline 1 total (retrieval only)",
              "Pipeline 2 total (retrieval + re-rank)",
              "LLM call, Pipeline 1 (avg)",
              "LLM call, Pipeline 2 (avg)"],
    "Avg ms": [ret_ms,
               rer_ms,
               ret_ms,
               ret_ms + rer_ms,
               np.mean([a for a, _ in llm_times]) * 1000,
               np.mean([b for _, b in llm_times]) * 1000],
}).round(1)
lat

,Stage,Avg ms
0,Bi-encoder retrieval (embed query + FAISS top-...,7.6
1,Cross-encoder re-rank of 10 pairs (Pipeline 2 ...,27.5
2,Pipeline 1 total (retrieval only),7.6
3,Pipeline 2 total (retrieval + re-rank),35.1
4,"LLM call, Pipeline 1 (avg)",2185.9
5,"LLM call, Pipeline 2 (avg)",3030.8
